# 环节 04 · 低秩适配公式（配套 Notebook）

> 配套长文：[环节04-低秩适配公式详解.md](./环节04-低秩适配公式详解.md) · 导航：[环节00](./环节00-总揽与环节导航.md)
> 定位：4×5 一层 20 个增量写成外积；λ 缩放不变；接到 7B 的 `2·d·r`。**纯标准库。**
> 公式约定与本库一致：`ΔW = B A`（B 高瘦，A 矮胖）。视频口播字母对调。

| 本 Notebook | 长文章节 | 验证什么 |
|---|---|---|
| §1 4×5 外积 | §2 | 20 格由 4+5 个数生成 |
| §2 尺度不变 | §2.2 | λB · (A/λ) 逐格相同，真 DOF=8 |
| §3 r 太大就没了 | §5 | 4×5 秩 2 已是 18 vs 20 |
| §4 接到 7B | §5 / 环节09 | 每投影 2·d·r，32 层 q/k/v/o ≈ 1678 万 |


In [ ]:
def fmt(mat, nd=1):
    return "\n".join("  [" + ", ".join(f"{v:{nd+5}.{nd}f}" for v in row) + "]" for row in mat)

def outer(u, v):
    return [[ui * vj for vj in v] for ui in u]

def scale_vec(x, g):
    return [g * t for t in x]

def equal(a, b, tol=1e-12):
    return all(abs(x - y) < tol for ra, rb in zip(a, b) for x, y in zip(ra, rb))


## 1. 一层 4→5：20 条边，秩 1 只需 9 个数（长文 §2）


In [ ]:
B = [2.0, 4.0, 6.0, 8.0]            # 4×1，本库的 B（高瘦）
A = [0.5, -1.0, 2.0, 0.0, 1.0]      # 1×5，本库的 A（矮胖）
dW = outer(B, A)

print("ΔW = B A  （4×5，看起来 20 个数）：\n" + fmt(dW))
print(f"\n字面参数：len(B)+len(A) = {len(B)+len(A)}")
print(f"全参增量：4×5 = {4*5}")
print("\n每一行都是 A 的倍数，每一列都是 B 的倍数 → 秩 1。")
print("前向：先 A·x（压到 1 维）再 ×B，从不物化这张 4×5。")


## 2. 尺度不变：9 个数其实是 8 个真自由度（长文 §2.2）

视频更正：右侧是乘积 `[a p, a q, …]`，不是 `[p/a, …]`。`(λB)·(A/λ)` 逐格相同。


In [ ]:
lam = 7.0
B2 = scale_vec(B, lam)
A2 = scale_vec(A, 1.0 / lam)
dW2 = outer(B2, A2)

print("λ = 7 之后的 ΔW：\n" + fmt(dW2))
print(f"\n与原 ΔW 逐格相等？ {equal(dW, dW2)}")

lam0 = 1.0 / B[0]
B3 = scale_vec(B, lam0)
A3 = scale_vec(A, 1.0 / lam0)
print(f"\n把 B[0] 缩放到 1（λ=1/a={lam0:.3f}）： B = {B3}")
print(f"ΔW 仍相等？ {equal(dW, outer(B3, A3))}")
print("\n真自由度 = 4+5-1 = 8。工程不去掉这 1 个数，报参数量仍用 9。")


## 3. r 相对层宽必须小（长文 §5）


In [ ]:
m, n = 4, 5
print(f"{'r':>4} {'字面 r(m+n)':>14} {'真 DOF r(m+n)-r²':>20} {'全参 mn':>10}  还省吗")
print("-" * 70)
for r in range(1, 5):
    literal = r * (m + n)
    true = literal - r * r
    full = m * n
    flag = "是" if literal < full else "轨快铺满 / 已不省"
    print(f"{r:>4} {literal:>14} {true:>20} {full:>10}  {flag}")

print("\n一般：真自由度 = r(m+n) - r²（可逆 r×r 混洗）。")
print("实现仍存 r(m+n) 个数；r 一接近 min(m,n)，LoRA 退化成贵一点的全参。")


## 4. 接到 7B 工程账（长文 §5 · [环节09](../transformer/环节09-训练管线详解.md) §4.3.3）

每投影字面参数 `2·d·r`（B 为 d×r，A 为 r×d）。注入 attention 的 q/k/v/o。


In [ ]:
d, L, r = 4096, 32, 16
per_proj = 2 * d * r
per_layer = per_proj * 4
total = per_layer * L
full_one = d * d

print(f"设定：hidden d={d}、{L} 层、注入 q/k/v/o、秩 r={r}\n")
print(f"  单投影全参     {full_one:>12,}")
print(f"  单投影 LoRA    {per_proj:>12,}   （本站公式 r(d+d) = {r*(d+d):,})")
print(f"  单投影省了     {full_one/per_proj:>12.1f}×")
print(f"  每层 4 投影    {per_layer:>12,}")
print(f"  {L} 层合计        {total:>12,}  ≈ {total/1e4:.0f} 万")
print(f"  占 7B          {total/7e9:.4%}")
print("\n与环节 09 手算对齐：≈ 1,678 万 ≈ 0.24%。")
print("几何课到此结束。显存账 / QLoRA / B 零初始化 / α/r → 环节 09。")
print("热插拔同一底座两份 (A,B) → 本目录 mvp.py。")
